# MeatLens Folder Inference Notebook (MobileNetV3Small Fold1 Seed42)

This notebook loads:
- `training_outputs/models/meatlens_mobilenetv3small_cross_rotation_fold1_seed42_cnn_only.h5`

Then runs batch prediction on images from:
1. A local folder path, or
2. An uploaded ZIP file of a folder

Output per image includes:
- predicted class
- confidence
- freshness score
- recommendation/scoring label


In [ ]:
import io
import json
import shutil
import zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input as preprocess_mobilenetv3

from IPython.display import display, clear_output

try:
    import ipywidgets as widgets
    IPYWIDGETS_AVAILABLE = True
except Exception:
    widgets = None
    IPYWIDGETS_AVAILABLE = False

print("TensorFlow:", tf.__version__)
print("ipywidgets available:", IPYWIDGETS_AVAILABLE)


TensorFlow: 2.10.0
ipywidgets available: True


In [ ]:
PROJECT_ROOT = Path.cwd()
MODELS_DIR = PROJECT_ROOT / "training_outputs" / "models"
INFERENCE_DIR = PROJECT_ROOT / "training_outputs" / "inference"
INFERENCE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / "meatlens_mobilenetv3small_cross_rotation_fold1_seed42_cnn_only.h5"
METADATA_PATH = MODELS_DIR / "meatlens_mobilenetv3small_cross_rotation_fold1_seed42_cnn_only_metadata.json"

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

if METADATA_PATH.exists():
    metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
else:
    metadata = {
        "backbone": "MobileNetV3Small",
        "input_size": [224, 224],
        "image_crop_mode": "center_crop",
        "label_order": ["fresh", "not fresh", "spoiled"],
    }

LABEL_ORDER = metadata.get("label_order", ["fresh", "not fresh", "spoiled"])
TARGET_SIZE = tuple(metadata.get("input_size", [224, 224]))
IMAGE_CROP_MODE = metadata.get("image_crop_mode", "center_crop")
BACKBONE_NAME = metadata.get("backbone", "MobileNetV3Small")

if BACKBONE_NAME != "MobileNetV3Small":
    raise ValueError(f"This notebook is fixed to MobileNetV3Small only, but metadata backbone is: {BACKBONE_NAME}")

print("MODEL_PATH:", MODEL_PATH)
print("METADATA_PATH:", METADATA_PATH)
print("BACKBONE_NAME:", BACKBONE_NAME)
print("TARGET_SIZE:", TARGET_SIZE)
print("IMAGE_CROP_MODE:", IMAGE_CROP_MODE)
print("LABEL_ORDER:", LABEL_ORDER)


MODEL_PATH: e:\Thesis Code\training_outputs\models\meatlens_mobilenetv3small_cross_rotation_fold1_seed42_cnn_only.h5
METADATA_PATH: e:\Thesis Code\training_outputs\models\meatlens_mobilenetv3small_cross_rotation_fold1_seed42_cnn_only_metadata.json
BACKBONE_NAME: MobileNetV3Small
TARGET_SIZE: (224, 224)
IMAGE_CROP_MODE: center_crop
LABEL_ORDER: ['fresh', 'not fresh', 'spoiled']


In [ ]:
# Deterministic preprocessing from new3.ipynb

def preprocess_resize(img: Image.Image, target_size=(224, 224)):
    return img.resize(target_size, Image.BILINEAR)


def preprocess_resize_pad(img: Image.Image, target_size=(224, 224), fill=(0, 0, 0)):
    canvas = Image.new("RGB", target_size, fill)
    copy = img.copy()
    copy.thumbnail(target_size, Image.BILINEAR)
    off_x = (target_size[0] - copy.size[0]) // 2
    off_y = (target_size[1] - copy.size[1]) // 2
    canvas.paste(copy, (off_x, off_y))
    return canvas


def preprocess_center_crop(img: Image.Image, target_size=(224, 224)):
    w, h = img.size
    side = min(w, h)
    left = (w - side) // 2
    top = (h - side) // 2
    crop = img.crop((left, top, left + side, top + side))
    return crop.resize(target_size, Image.BILINEAR)


def preprocess_roi_crop(img: Image.Image, row=None, target_size=(224, 224)):
    # In folder inference there is no ROI row, so match new3 fallback behavior.
    return preprocess_resize_pad(img, target_size=target_size)


def load_image_for_model(path, image_crop_mode="center_crop", target_size=(224, 224), row=None):
    if path is None:
        raise ValueError("Image path is None")

    img = Image.open(path).convert("RGB")
    original_np = np.array(img)

    mode = str(image_crop_mode).lower().strip()
    if mode == "resize":
        proc = preprocess_resize(img, target_size)
    elif mode == "resize_pad":
        proc = preprocess_resize_pad(img, target_size)
    elif mode == "center_crop":
        proc = preprocess_center_crop(img, target_size)
    elif mode == "roi_crop":
        proc = preprocess_roi_crop(img, row=row, target_size=target_size)
    else:
        raise ValueError(f"Unsupported IMAGE_CROP_MODE: {image_crop_mode}")

    proc_np = np.asarray(proc).astype(np.float32) / 255.0
    return proc_np, original_np


In [ ]:
# Scoring logic from new3.ipynb

def freshness_score(predicted_class, confidence):
    confidence = float(np.clip(confidence, 0.0, 1.0))
    if predicted_class == "fresh":
        return float(70 + (30 * confidence))
    if predicted_class == "not fresh":
        return float(40 + (20 * confidence))
    return float(max(0.0, 39 - (34 * confidence)))


def recommendation_from_score(score):
    if score >= 70:
        return "Good for Consumption"
    if score >= 40:
        return "Consume Immediately"
    return "Not Suitable"


In [ ]:
preprocess_fn = preprocess_mobilenetv3
model = tf.keras.models.load_model(MODEL_PATH, compile=False)

print("Model loaded successfully")


Model loaded successfully


In [ ]:
VALID_IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


def list_images_recursive(folder_path):
    folder = Path(folder_path)
    if not folder.exists() or not folder.is_dir():
        raise FileNotFoundError(f"Folder not found: {folder}")
    return sorted([p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in VALID_IMAGE_EXTS])


def predict_single_image(image_path):
    proc, _ = load_image_for_model(
        path=str(image_path),
        image_crop_mode=IMAGE_CROP_MODE,
        target_size=TARGET_SIZE,
        row=None,
    )
    x = preprocess_fn(proc * 255.0)
    x = np.expand_dims(x.astype(np.float32), axis=0)

    probs = model.predict(x, verbose=0)[0]
    pred_idx = int(np.argmax(probs))
    pred_class = LABEL_ORDER[pred_idx]
    confidence = float(probs[pred_idx])
    score = freshness_score(pred_class, confidence)
    recommendation = recommendation_from_score(score)

    row = {
        "image_path": str(image_path),
        "file_name": Path(image_path).name,
        "predicted_class": pred_class,
        "confidence": round(confidence, 6),
        "freshness_score": round(float(score), 2),
        "recommendation": recommendation,
    }
    for i, label in enumerate(LABEL_ORDER):
        row[f"prob_{label}"] = round(float(probs[i]), 6)
    return row


def predict_folder(folder_path, save_csv=True):
    image_paths = list_images_recursive(folder_path)
    if len(image_paths) == 0:
        raise ValueError(f"No supported image files found in: {folder_path}")

    rows = []
    errors = []
    for p in image_paths:
        try:
            rows.append(predict_single_image(p))
        except Exception as e:
            errors.append({"image_path": str(p), "error": str(e)})

    df = pd.DataFrame(rows)
    df = df.sort_values(["freshness_score", "confidence"], ascending=[False, False]).reset_index(drop=True)

    summary = {
        "total_images": int(len(image_paths)),
        "successful_predictions": int(len(df)),
        "failed_predictions": int(len(errors)),
        "mean_confidence": float(df["confidence"].mean()) if len(df) else None,
        "mean_freshness_score": float(df["freshness_score"].mean()) if len(df) else None,
        "class_counts": df["predicted_class"].value_counts().to_dict() if len(df) else {},
    }

    out_csv = None
    if save_csv and len(df) > 0:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        out_csv = INFERENCE_DIR / f"folder_inference_{ts}.csv"
        df.to_csv(out_csv, index=False)

    return df, errors, summary, out_csv


In [ ]:
def _normalize_uploaded_items(upload_value):
    items = []

    if isinstance(upload_value, dict):
        for name, payload in upload_value.items():
            content = payload.get("content") if isinstance(payload, dict) else None
            if content is not None:
                items.append({"name": name, "content": content})
        return items

    if isinstance(upload_value, (list, tuple)):
        for payload in upload_value:
            if isinstance(payload, dict):
                name = payload.get("name", "uploaded.zip")
                content = payload.get("content")
                if content is not None:
                    items.append({"name": name, "content": content})
        return items

    return items


def extract_uploaded_zip_to_temp(upload_item):
    temp_root = INFERENCE_DIR / "uploaded_zips"
    temp_root.mkdir(parents=True, exist_ok=True)

    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    work_dir = temp_root / f"zip_{stamp}"
    if work_dir.exists():
        shutil.rmtree(work_dir)
    work_dir.mkdir(parents=True, exist_ok=True)

    zip_path = work_dir / upload_item["name"]
    zip_path.write_bytes(upload_item["content"])

    extract_dir = work_dir / "extracted"
    extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(io.BytesIO(upload_item["content"]), "r") as zf:
        zf.extractall(extract_dir)

    return extract_dir


In [ ]:
if IPYWIDGETS_AVAILABLE:
    upload_widget = widgets.FileUpload(
        accept=".zip",
        multiple=False,
        description="Upload ZIP"
    )
    folder_text = widgets.Text(
        value="",
        placeholder="Example: E:/Thesis Code/Pork Shoulder - sample 1",
        description="Folder path:",
        layout=widgets.Layout(width="700px"),
    )
    run_button = widgets.Button(description="Run Inference", button_style="success")
    output_area = widgets.Output()

    display(widgets.HTML("<b>Option A:</b> Upload a ZIP file containing your folder of images."))
    display(upload_widget)
    display(widgets.HTML("<b>Option B:</b> Enter a local folder path and run."))
    display(folder_text)
    display(run_button)
    display(output_area)

    def _run_inference(_):
        with output_area:
            clear_output(wait=True)
            try:
                folder_to_use = None
                uploaded_items = _normalize_uploaded_items(upload_widget.value)

                if len(uploaded_items) > 0:
                    print("Using uploaded ZIP file...")
                    folder_to_use = extract_uploaded_zip_to_temp(uploaded_items[0])
                    print("Extracted to:", folder_to_use)
                elif folder_text.value.strip():
                    folder_to_use = Path(folder_text.value.strip())
                    print("Using local folder:", folder_to_use)
                else:
                    print("Please upload a ZIP file or enter a folder path.")
                    return

                df, errors, summary, out_csv = predict_folder(folder_to_use, save_csv=True)

                print("\nSummary:")
                print(json.dumps(summary, indent=2))

                if out_csv is not None:
                    print("\nSaved CSV:", out_csv)

                print("\nTop predictions:")
                display(df.head(20))

                if len(errors) > 0:
                    print("\nErrors:")
                    display(pd.DataFrame(errors).head(20))

            except Exception as e:
                print("Error:", e)

    run_button.on_click(_run_inference)
else:
    print("ipywidgets is not installed in this kernel.")
    print("Use manual mode in the next cell.")


HTML(value='<b>Option A:</b> Upload a ZIP file containing your folder of images.')

FileUpload(value=(), accept='.zip', description='Upload ZIP')

HTML(value='<b>Option B:</b> Enter a local folder path and run.')

Text(value='', description='Folder path:', layout=Layout(width='700px'), placeholder='Example: E:/Thesis Code/…

Button(button_style='success', description='Run Inference', style=ButtonStyle())

Output()

In [ ]:
# Manual mode (works even without widgets)
# Set any folder containing meat images, then run this cell.

MANUAL_FOLDER_PATH = ""

if MANUAL_FOLDER_PATH:
    df, errors, summary, out_csv = predict_folder(MANUAL_FOLDER_PATH, save_csv=True)
    print(json.dumps(summary, indent=2))
    if out_csv is not None:
        print("Saved CSV:", out_csv)
    display(df.head(30))
    if len(errors) > 0:
        display(pd.DataFrame(errors).head(20))
else:
    print("Set MANUAL_FOLDER_PATH first.")
